In [ ]:
import pandas as pd

In [2]:
pd.__version__

'3.0.3'

In [3]:

df = pd.read_pickle('../data/LSWMD_slimmed.pkl')
df.info()

<class 'pandas.DataFrame'>
Index: 12822 entries, 19 to 811447
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   waferMap  12822 non-null  object
 1   label     12822 non-null  str   
dtypes: object(1), str(1)
memory usage: 300.5+ KB


In [4]:
df.head()

,waferMap,label
19,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Loc
36,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Edge-Loc
39,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Edge-Loc
40,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Loc
41,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",Edge-Loc


In [5]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


TARGET_SIZE = (96, 96) 

def wafertoscale(wafer_matrix): #preprocesses single wafermap
   
    tensor = tf.convert_to_tensor(wafer_matrix, dtype=tf.float32)  
    tensor = tf.expand_dims(tensor, axis=-1) # Add the channel dimension: (Height, Width) -> (Height, Width, 1)
    
    # Resize with pad:
    resized_tensor = tf.image.resize_with_pad(
        tensor, 
        target_height=TARGET_SIZE[0], 
        target_width=TARGET_SIZE[1], 
        method=tf.image.ResizeMethod.NEAREST_NEIGHBOR
    )
    
    return resized_tensor.numpy()

# Apply preprocessing to all wafers
df['processed'] = np.stack([wafertoscale(w) for w in df['waferMap']])

print(f"Final Image Data Shape: {df['processed'].shape}") 
# Expected Output: (12822, 96, 96, 1)

2026-05-13 19:48:15.133885: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# Convert string labels ('Loc', 'Edge-Loc', etc.) to integers (0, 1, 2...)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df['label'])

# Save the number of classes for your final Dense layer later
num_classes = len(label_encoder.classes_)
print(f"Found {num_classes} unique classes: {label_encoder.classes_}")

# Perform the Train/Test split (80% training, 20% testing)
# 'stratify=y_encoded' ensures both splits have a proportional amount of each defect class
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, 
    y_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded
)